# Chapter 8.5: User Modeling at Scale

## Learning Objectives

By the end of this notebook, you will be able to:

1. Design long-term user profiles that capture stable preferences and demographics
2. Implement short-term/real-time interest detection from session-level behavior
3. Apply user segmentation through clustering and persona-based modeling
4. Build hierarchical user models combining global preferences with context-specific interests
5. Understand privacy-preserving user modeling with differential privacy
6. Evaluate user models on preference prediction tasks
7. Build a complete hierarchical user model with long-term and short-term components

## Prerequisites

- Chapter 8.1-8.4: Embedding and feature fundamentals
- Understanding of sequence models (RNNs, attention)
- Basic clustering knowledge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part8/chapter_8.5_user_modeling.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part8/chapter_8.5_user_modeling.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8')
print(f"PyTorch version: {torch.__version__}")

## 1. Long-Term User Profiles

Long-term user profiles capture **stable preferences** that don't change much over time:

- Demographic features (age, gender, location)
- Aggregated category preferences over months/years
- Price sensitivity
- Brand affinities

These are typically updated in batch (daily or weekly) and stored in the feature store's online tier.

$$\mathbf{u}_{\text{long}} = \text{MLP}(\text{concat}(\mathbf{e}_{\text{demo}}, \mathbf{h}_{\text{agg}}, \mathbf{p}_{\text{pref}}))$$

where $\mathbf{h}_{\text{agg}}$ are aggregated behavioral features and $\mathbf{p}_{\text{pref}}$ are learned preference vectors.

> **\U0001f4a1 Concept:** Long-term profiles answer the question "What kind of user is this?" They capture identity-level preferences that are stable across sessions.

In [ ]:
class LongTermUserProfile(nn.Module):
    """Long-term user modeling from aggregated features."""
    
    def __init__(
        self,
        num_users: int,
        num_categories: int,
        num_demographics: int,
        embedding_dim: int,
    ):
        super().__init__()
        self.user_id_emb = nn.Embedding(num_users, embedding_dim)
        
        # Category preference encoder
        self.category_encoder = nn.Sequential(
            nn.Linear(num_categories, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim),
        )
        
        # Demographic encoder
        self.demo_encoder = nn.Sequential(
            nn.Linear(num_demographics, 32),
            nn.ReLU(),
            nn.Linear(32, embedding_dim),
        )
        
        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(embedding_dim * 3, embedding_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(embedding_dim * 2, embedding_dim),
        )
    
    def forward(
        self,
        user_ids: torch.Tensor,
        category_prefs: torch.Tensor,  # Aggregated category interaction counts
        demographics: torch.Tensor,
    ) -> torch.Tensor:
        id_emb = self.user_id_emb(user_ids)
        cat_emb = self.category_encoder(category_prefs)
        demo_emb = self.demo_encoder(demographics)
        
        combined = torch.cat([id_emb, cat_emb, demo_emb], dim=-1)
        return self.fusion(combined)

# Generate synthetic user data
num_users = 5000
num_categories = 20
num_demographics = 8
embedding_dim = 32

# Synthetic category preferences (log-transformed interaction counts)
raw_cat_counts = np.random.exponential(5, (num_users, num_categories))
# Users have 2-3 dominant categories
for u in range(num_users):
    dominant_cats = np.random.choice(num_categories, size=3, replace=False)
    raw_cat_counts[u, dominant_cats] *= 5

category_prefs = torch.tensor(np.log1p(raw_cat_counts), dtype=torch.float32)

# Synthetic demographics
demographics = torch.randn(num_users, num_demographics)

# Build and test model
long_term_model = LongTermUserProfile(num_users, num_categories, num_demographics, embedding_dim)

test_users = torch.arange(0, 10)
with torch.no_grad():
    profiles = long_term_model(test_users, category_prefs[test_users], demographics[test_users])

print(f"Long-term profile shape: {profiles.shape}")
print(f"Model parameters: {sum(p.numel() for p in long_term_model.parameters()):,}")

## 2. Short-Term Interest Detection

Short-term interests capture what the user wants **right now**, based on their current session:

$$\mathbf{u}_{\text{short}} = \text{Attention}(\mathbf{q}, [\mathbf{e}_{a_1}, \mathbf{e}_{a_2}, \ldots, \mathbf{e}_{a_T}])$$

where $a_1, \ldots, a_T$ are recent actions and $\mathbf{q}$ is a learned query or the target item embedding.

Key patterns:
- **Session-level GRU/LSTM**: Sequential modeling of actions within a session
- **Self-attention**: Capture dependencies between recent actions
- **Target attention**: Weight recent actions by relevance to the candidate item

> **\u26a0\ufe0f Common Pitfall:** Short-term models can be noisy. A user who accidentally clicks on one sports article shouldn't suddenly get all sports recommendations. Use attention weights to filter noise.

In [ ]:
class ShortTermInterestModel(nn.Module):
    """Session-level interest detection using self-attention."""
    
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_heads: int = 4,
        max_seq_len: int = 50,
    ):
        super().__init__()
        self.item_emb = nn.Embedding(num_items + 1, embedding_dim, padding_idx=0)  # 0 = padding
        self.pos_emb = nn.Embedding(max_seq_len, embedding_dim)
        
        # Self-attention for sequence modeling
        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            batch_first=True,
        )
        
        # Target attention: how relevant is each past action to the candidate?
        self.target_attention = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        
        self.norm = nn.LayerNorm(embedding_dim)
    
    def forward(
        self,
        action_sequence: torch.Tensor,  # (batch, seq_len) item IDs
        target_item: Optional[torch.Tensor] = None,  # (batch,) candidate item ID
    ) -> torch.Tensor:
        batch_size, seq_len = action_sequence.shape
        
        # Embed actions with positional encoding
        positions = torch.arange(seq_len, device=action_sequence.device).unsqueeze(0).expand(batch_size, -1)
        action_embs = self.item_emb(action_sequence) + self.pos_emb(positions)
        
        # Padding mask
        padding_mask = (action_sequence == 0)  # True where padded
        
        # Self-attention
        attended, attn_weights = self.self_attention(
            action_embs, action_embs, action_embs,
            key_padding_mask=padding_mask,
        )
        attended = self.norm(attended + action_embs)  # Residual
        
        if target_item is not None:
            # Target-aware attention
            target_emb = self.item_emb(target_item).unsqueeze(1).expand(-1, seq_len, -1)
            concat = torch.cat([attended, target_emb], dim=-1)
            weights = self.target_attention(concat).squeeze(-1)  # (batch, seq_len)
            weights = weights.masked_fill(padding_mask, float('-inf'))
            weights = F.softmax(weights, dim=-1)
            user_repr = (attended * weights.unsqueeze(-1)).sum(dim=1)
        else:
            # Mean pooling (ignoring padding)
            mask = (~padding_mask).float().unsqueeze(-1)
            user_repr = (attended * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        
        return user_repr

# Test with synthetic session data
num_items = 1000
short_term_model = ShortTermInterestModel(num_items, embedding_dim, num_heads=4, max_seq_len=20)

# Generate synthetic sessions
batch_size = 32
seq_len = 15
sessions = torch.randint(1, num_items + 1, (batch_size, seq_len))
# Add some padding
for i in range(batch_size):
    pad_start = np.random.randint(5, seq_len)
    sessions[i, pad_start:] = 0

target_items = torch.randint(1, num_items + 1, (batch_size,))

with torch.no_grad():
    short_repr = short_term_model(sessions, target_items)
    short_repr_no_target = short_term_model(sessions)

print(f"Short-term representation shape: {short_repr.shape}")
print(f"Model parameters: {sum(p.numel() for p in short_term_model.parameters()):,}")

## 3. User Segmentation and Persona Modeling

**User segmentation** groups users into clusters (personas) that share behavioral patterns. This helps with:

- Cold-start: assign new users to the best matching persona
- Exploration: recommend based on what similar users liked
- Analysis: understand user archetypes for product decisions

$$p(\text{persona} | \text{user}) = \text{softmax}(W \cdot \mathbf{u} + b)$$

> **\U0001f4a1 Concept:** Instead of hard clustering, **soft persona assignment** lets a user belong to multiple personas with different weights. A user might be 60% "deal hunter" and 40% "brand loyalist."

In [ ]:
class PersonaModel(nn.Module):
    """Soft persona assignment for user segmentation."""
    
    def __init__(
        self,
        input_dim: int,
        num_personas: int,
        persona_dim: int,
    ):
        super().__init__()
        self.num_personas = num_personas
        
        # Persona assignment network
        self.persona_classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_personas),
        )
        
        # Learnable persona embeddings
        self.persona_embs = nn.Parameter(torch.randn(num_personas, persona_dim) * 0.1)
    
    def forward(self, user_features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # Soft persona assignment
        logits = self.persona_classifier(user_features)
        weights = F.softmax(logits, dim=-1)  # (batch, num_personas)
        
        # Weighted combination of persona embeddings
        persona_repr = torch.matmul(weights, self.persona_embs)  # (batch, persona_dim)
        
        return persona_repr, weights

# Create user feature matrix from category preferences
user_feature_matrix = category_prefs  # (5000, 20)

# Train persona model with reconstruction objective
num_personas = 8
persona_model = PersonaModel(num_categories, num_personas, embedding_dim)
decoder = nn.Sequential(
    nn.Linear(embedding_dim, 64),
    nn.ReLU(),
    nn.Linear(64, num_categories),
)

optimizer = torch.optim.Adam(
    list(persona_model.parameters()) + list(decoder.parameters()), lr=0.005
)

for epoch in range(100):
    perm = torch.randperm(num_users)
    total_loss = 0
    for i in range(0, num_users, 256):
        idx = perm[i:i+256]
        features = user_feature_matrix[idx]
        
        persona_repr, weights = persona_model(features)
        reconstructed = decoder(persona_repr)
        
        # Reconstruction loss + entropy regularization
        recon_loss = F.mse_loss(reconstructed, features)
        # Encourage diverse persona usage
        avg_weights = weights.mean(dim=0)
        entropy = -(avg_weights * torch.log(avg_weights + 1e-8)).sum()
        loss = recon_loss - 0.1 * entropy  # Maximize entropy
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += recon_loss.item()

# Analyze personas
with torch.no_grad():
    _, all_weights = persona_model(user_feature_matrix)
    hard_assignments = all_weights.argmax(dim=1).numpy()

print("Persona Distribution:")
for p in range(num_personas):
    count = (hard_assignments == p).sum()
    print(f"  Persona {p}: {count} users ({100*count/num_users:.1f}%)")

In [ ]:
# Visualize persona analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Persona assignment distribution
persona_counts = [(hard_assignments == p).sum() for p in range(num_personas)]
ax1.bar(range(num_personas), persona_counts, color='steelblue', edgecolor='black')
ax1.set_xlabel('Persona', fontsize=12)
ax1.set_ylabel('Number of Users', fontsize=12)
ax1.set_title('User Distribution Across Personas', fontsize=13, fontweight='bold')

# Plot 2: Average category preferences per persona
persona_cat_prefs = np.zeros((num_personas, num_categories))
for p in range(num_personas):
    mask = hard_assignments == p
    if mask.sum() > 0:
        persona_cat_prefs[p] = category_prefs[mask].numpy().mean(axis=0)

im = ax2.imshow(persona_cat_prefs[:, :10], aspect='auto', cmap='YlOrRd')
ax2.set_xlabel('Category (top 10)', fontsize=12)
ax2.set_ylabel('Persona', fontsize=12)
ax2.set_title('Category Preferences by Persona', fontsize=13, fontweight='bold')
ax2.set_yticks(range(num_personas))
plt.colorbar(im, ax=ax2, label='Avg Preference Score')

plt.tight_layout()
plt.savefig('user_personas.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Hierarchical User Model

A **hierarchical user model** combines multiple levels of user understanding:

$$\mathbf{u} = \text{Fusion}(\mathbf{u}_{\text{long}}, \mathbf{u}_{\text{short}}, \mathbf{u}_{\text{persona}}, \mathbf{u}_{\text{context}})$$

where:
- $\mathbf{u}_{\text{long}}$: stable preferences (updated daily)
- $\mathbf{u}_{\text{short}}$: session-level interests (updated per request)
- $\mathbf{u}_{\text{persona}}$: persona-level representation
- $\mathbf{u}_{\text{context}}$: time, device, location context

> **\U0001f511 Pro Tip:** Use **attention-based fusion** where the fusion weights depend on the query context. During a quick mobile browse, short-term interests should dominate. During a deliberate desktop shopping session, long-term preferences matter more.

In [ ]:
class HierarchicalUserModel(nn.Module):
    """Combines long-term, short-term, and persona representations."""
    
    def __init__(
        self,
        num_users: int,
        num_items: int,
        num_categories: int,
        num_demographics: int,
        num_personas: int,
        context_dim: int,
        embedding_dim: int,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        # Sub-models
        self.long_term = LongTermUserProfile(
            num_users, num_categories, num_demographics, embedding_dim
        )
        self.short_term = ShortTermInterestModel(
            num_items, embedding_dim, num_heads=4, max_seq_len=20
        )
        self.persona = PersonaModel(num_categories, num_personas, embedding_dim)
        
        # Context encoder
        self.context_encoder = nn.Sequential(
            nn.Linear(context_dim, 32),
            nn.ReLU(),
            nn.Linear(32, embedding_dim),
        )
        
        # Attention-based fusion
        self.fusion_query = nn.Linear(embedding_dim, embedding_dim)
        self.fusion_key = nn.Linear(embedding_dim, embedding_dim)
        self.fusion_value = nn.Linear(embedding_dim, embedding_dim)
        
        # Output projection
        self.output_proj = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim),
        )
    
    def forward(
        self,
        user_ids: torch.Tensor,
        category_prefs: torch.Tensor,
        demographics: torch.Tensor,
        action_sequence: torch.Tensor,
        context_features: torch.Tensor,
        target_item: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        batch_size = user_ids.shape[0]
        
        # Compute component representations
        long_repr = self.long_term(user_ids, category_prefs, demographics)
        short_repr = self.short_term(action_sequence, target_item)
        persona_repr, persona_weights = self.persona(category_prefs)
        context_repr = self.context_encoder(context_features)
        
        # Stack representations: (batch, 4, dim)
        components = torch.stack([long_repr, short_repr, persona_repr, context_repr], dim=1)
        
        # Attention-based fusion (context as query)
        query = self.fusion_query(context_repr).unsqueeze(1)  # (batch, 1, dim)
        keys = self.fusion_key(components)  # (batch, 4, dim)
        values = self.fusion_value(components)
        
        attn_scores = torch.bmm(query, keys.transpose(1, 2)) / (self.embedding_dim ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)  # (batch, 1, 4)
        
        fused = torch.bmm(attn_weights, values).squeeze(1)  # (batch, dim)
        user_repr = self.output_proj(fused)
        
        # Debug info
        info = {
            'fusion_weights': attn_weights.squeeze(1).detach(),
            'persona_weights': persona_weights.detach(),
            'component_norms': components.norm(dim=-1).detach(),
        }
        
        return user_repr, info

# Build and test
num_items = 1000
context_dim = 5  # hour_of_day, day_of_week, device, etc.

hierarchical_model = HierarchicalUserModel(
    num_users=num_users,
    num_items=num_items,
    num_categories=num_categories,
    num_demographics=num_demographics,
    num_personas=8,
    context_dim=context_dim,
    embedding_dim=embedding_dim,
)

# Generate test data
batch_size = 16
test_user_ids = torch.randint(0, num_users, (batch_size,))
test_sessions = torch.randint(1, num_items + 1, (batch_size, 10))
test_context = torch.randn(batch_size, context_dim)
test_target = torch.randint(1, num_items + 1, (batch_size,))

with torch.no_grad():
    user_repr, info = hierarchical_model(
        test_user_ids,
        category_prefs[test_user_ids],
        demographics[test_user_ids],
        test_sessions,
        test_context,
        test_target,
    )

print(f"User representation shape: {user_repr.shape}")
print(f"Total parameters: {sum(p.numel() for p in hierarchical_model.parameters()):,}")
print(f"\nFusion weights (long, short, persona, context):")
print(f"  Mean: {info['fusion_weights'].mean(dim=0).numpy().round(3)}")
print(f"  Std:  {info['fusion_weights'].std(dim=0).numpy().round(3)}")

## 5. Privacy-Preserving User Modeling

**Differential Privacy (DP)** provides formal guarantees that individual user data cannot be reconstructed from the model.

For user embeddings with DP-SGD:

$$\tilde{g} = \frac{1}{B} \sum_{i=1}^{B} \text{clip}(g_i, C) + \mathcal{N}(0, \sigma^2 C^2 I)$$

where $C$ is the clipping norm and $\sigma$ controls the privacy level.

> **\U0001f4a1 Concept:** The privacy-utility trade-off is fundamental. More noise means stronger privacy but worse recommendations. The key insight is that user embeddings for popular behavior patterns can be learned with strong DP, while rare behaviors require relaxed privacy or on-device processing.

In [ ]:
class DPUserEmbedding(nn.Module):
    """User embedding with simulated differential privacy."""
    
    def __init__(
        self,
        num_users: int,
        embedding_dim: int,
        clip_norm: float = 1.0,
        noise_multiplier: float = 0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(num_users, embedding_dim)
        self.clip_norm = clip_norm
        self.noise_multiplier = noise_multiplier
        self.embedding_dim = embedding_dim
    
    def forward(self, user_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(user_ids)
        
        if self.training:
            # Add calibrated noise during training (DP mechanism)
            noise_scale = self.noise_multiplier * self.clip_norm
            noise = torch.randn_like(emb) * noise_scale
            emb = emb + noise
        
        return emb
    
    def clip_gradients(self):
        """Clip per-sample gradients to bound sensitivity."""
        if self.embedding.weight.grad is not None:
            grad = self.embedding.weight.grad
            grad_norms = grad.norm(dim=1, keepdim=True)
            clip_factor = (self.clip_norm / grad_norms.clamp(min=self.clip_norm))
            self.embedding.weight.grad = grad * clip_factor

# Compare utility at different noise levels
noise_levels = [0.0, 0.05, 0.1, 0.5, 1.0, 2.0]
quality_scores = []

for noise in noise_levels:
    dp_emb = DPUserEmbedding(100, 32, clip_norm=1.0, noise_multiplier=noise)
    dp_emb.train()
    
    # Measure embedding quality: can we distinguish users?
    test_ids = torch.arange(0, 20)
    
    # Multiple noisy lookups for same users
    embeddings_list = []
    for _ in range(10):
        with torch.no_grad():
            embeddings_list.append(dp_emb(test_ids))
    
    # Measure consistency (how similar are repeated lookups for same user)
    stacked = torch.stack(embeddings_list)  # (10, 20, 32)
    mean_emb = stacked.mean(dim=0)  # (20, 32)
    
    # Avg cosine similarity between lookups of same user
    consistency = 0
    for i in range(10):
        for j in range(i+1, 10):
            cos_sim = F.cosine_similarity(stacked[i], stacked[j], dim=1).mean()
            consistency += cos_sim.item()
    consistency /= 45  # 10 choose 2
    
    quality_scores.append(consistency)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(noise_levels, quality_scores, 'o-', linewidth=2, markersize=10, color='steelblue')
ax.set_xlabel('Noise Multiplier (higher = more private)', fontsize=12)
ax.set_ylabel('Embedding Consistency', fontsize=12)
ax.set_title('Privacy-Utility Trade-off for User Embeddings', fontsize=13, fontweight='bold')
ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='Acceptable threshold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('dp_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Exercises

### \U0001f3cb\ufe0f Exercise 1: Build Multi-Interest User Model

Implement a user model that captures **multiple diverse interests** using capsule-style routing.

In [ ]:
# Exercise 1: Multi-Interest User Model
# TODO: Implement a model that extracts K diverse interest vectors from user behavior

class MultiInterestUserModel(nn.Module):
    """
    Extract K diverse interest vectors from user's action sequence.
    Uses attention with K learnable queries to extract different interests.
    
    Inspired by MIND (Li et al., Alibaba, 2019).
    """
    
    def __init__(
        self, num_items: int, embedding_dim: int,
        num_interests: int = 4
    ):
        super().__init__()
        # TODO: Implement
        # Create item embeddings
        # Create K learnable interest queries
        # Each query attends to the action sequence to extract one interest
        pass
    
    def forward(
        self, action_sequence: torch.Tensor
    ) -> torch.Tensor:
        """
        Returns: (batch, num_interests, embedding_dim)
        """
        # TODO: Implement
        pass

# Test: verify that different interest vectors capture different aspects

### \U0001f3cb\ufe0f Exercise 2: Context-Adaptive Fusion

Implement a fusion mechanism where the weights between long-term and short-term models adapt to the context (time of day, device type, etc.).

In [ ]:
# Exercise 2: Context-Adaptive Fusion
# TODO: Implement context-dependent fusion of long-term and short-term interests

class ContextAdaptiveFusion(nn.Module):
    """
    Fuses long-term and short-term representations based on context:
    - During quick mobile browse -> more weight on short-term
    - During deliberate desktop shopping -> more weight on long-term
    - Late night -> different pattern than daytime
    """
    
    def __init__(self, user_dim: int, context_dim: int):
        super().__init__()
        # TODO: Implement a gating network that takes context features
        # and outputs weights for long-term vs short-term
        pass
    
    def forward(
        self,
        long_term_repr: torch.Tensor,
        short_term_repr: torch.Tensor,
        context: torch.Tensor,
    ) -> torch.Tensor:
        # TODO: Implement
        pass

# Test with different context scenarios

### \U0001f3cb\ufe0f Exercise 3: User Interest Evolution Tracking

Build a system that tracks how user interests evolve over time and detects interest shifts.

In [ ]:
# Exercise 3: Interest Evolution Tracker
# TODO: Track how a user's interests change over time

class InterestEvolutionTracker:
    """
    Tracks user interest evolution:
    1. Maintain a sliding window of user representations
    2. Detect significant shifts (cosine distance > threshold)
    3. Visualize interest trajectory
    """
    
    def __init__(self, window_size: int = 10, shift_threshold: float = 0.3):
        # TODO: Initialize
        pass
    
    def update(self, user_id: str, representation: torch.Tensor, timestamp: float):
        # TODO: Add new representation and check for shift
        pass
    
    def detect_shift(self, user_id: str) -> bool:
        # TODO: Compare recent representations with historical
        pass
    
    def get_evolution(self, user_id: str) -> List[Tuple[float, torch.Tensor]]:
        # TODO: Return the history of (timestamp, representation) pairs
        pass

# Simulate a user who gradually shifts from one interest to another

## Summary

In this notebook, we built a comprehensive user modeling system:

1. **Long-term profiles**: Stable preferences from demographics and aggregated behavior
2. **Short-term interests**: Session-level signals using attention over recent actions
3. **User segmentation**: Soft persona assignment for understanding user archetypes
4. **Hierarchical models**: Attention-based fusion of multiple user representations
5. **Privacy-preserving**: Differential privacy mechanisms for user embeddings

### Key References

- Li et al. "Multi-Interest Network with Dynamic Routing (MIND)" (Alibaba, CIKM 2019)
- Pi et al. "Search-based User Interest Modeling (SIM)" (Alibaba, CIKM 2020)
- Zhou et al. "Deep Interest Network (DIN)" (Alibaba, KDD 2018)
- Zhou et al. "Deep Interest Evolution Network (DIEN)" (Alibaba, AAAI 2019)
- Abadi et al. "Deep Learning with Differential Privacy" (Google, CCS 2016)

### Next Steps

In Chapter 8.6, we will explore **Item Understanding** -- building rich multimodal representations of items.